# SnapKV on SGLang — notebook compression demo

Reproduction of the upstream SnapKV notebook
([`FasterDecoding/SnapKV notebooks/example.ipynb`](https://github.com/FasterDecoding/SnapKV/blob/main/notebooks/example.ipynb))
against **our SGLang SnapKV port** (not the HuggingFace monkeypatch).

We feed the full SnapKV paper (`data/snapkv.txt`) as the prefill — a **long
input** — and ask one question at the end — a **short output**:

> `\n What is the repository of SnapKV?`

SnapKV compresses the prompt KV to `max_capacity_prompt` right after prefill and
physically frees the rest. If it keeps the right tokens, the model still answers
correctly from the shrunken cache.

Run this notebook from `SnapKV/benchmark/`. It launches its own SGLang server
(model `Mistral-7B-Instruct-v0.2`, the notebook's own choice — 32k context, so
the full ~17k-token article fits), queries it, prints the answer and the
server-side compaction log, then shuts the server down.

In [1]:
import json, os, pathlib, signal, subprocess, time, urllib.request

BENCH = pathlib.Path.cwd()                 # .../SnapKV/benchmark
REPO = BENCH.parents[1]                     # repo root
MODEL = os.environ.get("MODEL", "/data/model/Mistral-7B-Instruct-v0.2")
PORT = int(os.environ.get("PORT", "30060"))
BUDGET = 1024                               # max_capacity_prompt (KV kept per request)
GPU = os.environ.get("CUDA_VISIBLE_DEVICES", "0")
SERVER_LOG = "/tmp/snapkv_notebook_server.log"
print("repo   :", REPO)
print("model  :", MODEL)
print("port   :", PORT, "| budget (max_capacity_prompt):", BUDGET)

repo   : /home/sigma/github/sglang-compress
model  : /data/model/Mistral-7B-Instruct-v0.2
port   : 30060 | budget (max_capacity_prompt): 1024


In [2]:
# Launch a SnapKV-enabled SGLang server (SnapKV requires the eager / radix-off /
# page_size=1 / chunked-prefill-off config; launch_server.sh sets those flags).
env = dict(os.environ, CUDA_VISIBLE_DEVICES=GPU, PORT=str(PORT), MODEL=MODEL, MEM_FRAC="0.85")
log = open(SERVER_LOG, "w")
proc = subprocess.Popen(
    ["bash", "./launch_server.sh", "snapkv", str(BUDGET)],
    stdout=log, stderr=subprocess.STDOUT, env=env, cwd=str(BENCH),
)

ready = False
for _ in range(120):
    txt = pathlib.Path(SERVER_LOG).read_text(errors="ignore")
    if "The server is fired up and ready to roll" in txt:
        ready = True
        break
    if proc.poll() is not None:
        break
    time.sleep(3)
print("server pid:", proc.pid, "| ready:", ready)
assert ready, "server did not become ready — see " + SERVER_LOG

server pid: 1905335 | ready: True


In [3]:
# Build the prompt exactly like the upstream notebook: full article + question.
content = pathlib.Path("data/snapkv.txt").read_text().strip()
question = "\n What is the repository of SnapKV?"
print("article words :", len(content.split()))
print("question      :", repr(question))

article words : 6323
question      : '\n What is the repository of SnapKV?'


In [4]:
# Query the SnapKV server (chat template applied via /v1/chat/completions).
body = json.dumps({
    "model": MODEL,
    "messages": [{"role": "user", "content": content + question}],
    "temperature": 0.0,
    "max_tokens": 200,
}).encode()
req = urllib.request.Request(
    f"http://127.0.0.1:{PORT}/v1/chat/completions",
    data=body, headers={"Content-Type": "application/json"},
)
out = json.loads(urllib.request.urlopen(req, timeout=600).read())
usage = out["usage"]
answer = out["choices"][0]["message"]["content"]

print("prompt_tokens     :", usage["prompt_tokens"])
print("completion_tokens :", usage["completion_tokens"])
print("=== ANSWER (SnapKV on) ===")
print(answer)

prompt_tokens     : 17395
completion_tokens : 28
=== ANSWER (SnapKV on) ===
 The repository of SnapKV is available at <https://github.com/FasterDecoding/SnapKV>.


In [5]:
# Proof of real physical eviction: the server logs each SnapKV compaction.
for line in pathlib.Path(SERVER_LOG).read_text(errors="ignore").splitlines():
    if "SnapKV compacted" in line:
        print(line.strip())

[2026-07-06 22:40:09] SnapKV compacted req_pool_idx=2: prompt 17395 -> 1024 slots (freed 16371)


In [6]:
# Shut the server down.
proc.send_signal(signal.SIGINT)
try:
    proc.wait(timeout=20)
except subprocess.TimeoutExpired:
    proc.kill()
print("server stopped (returncode=%s)" % proc.returncode)

server stopped (returncode=0)
